# CatBoost – Fraud Detection (BankSim)
Cấu trúc: Load data → Baseline → Optuna → Retrain → Evaluation → Visualization

## 1. Import & Setup

In [ ]:
import joblib
import numpy as np
import pandas as pd
import optuna
import matplotlib.pyplot as plt
import seaborn as sns

from catboost import CatBoostClassifier
from sklearn.metrics import (
    f1_score, recall_score, precision_score,
    roc_auc_score, average_precision_score,
    confusion_matrix, classification_report,
    precision_recall_curve
)

optuna.logging.set_verbosity(optuna.logging.WARNING)
SEED = 42

In [ ]:
from sklearn.metrics import log_loss
import shap

def sample_xy(X, y, max_samples=30000, random_state=42):
    if len(y) <= max_samples:
        return X, np.asarray(y)

    rng = np.random.default_rng(random_state)
    idx = np.sort(rng.choice(len(y), size=max_samples, replace=False))

    if hasattr(X, "iloc"):
        X_sample = X.iloc[idx].copy()
    else:
        X_sample = X[idx]

    y_array = np.asarray(y)
    return X_sample, y_array[idx]


def build_checkpoints(total_steps, max_points=60):
    if total_steps <= max_points:
        return list(range(1, total_steps + 1))

    checkpoints = np.linspace(1, total_steps, num=max_points, dtype=int)
    return sorted(set(int(step) for step in checkpoints))


def plot_loss_curve(loss_df, title):
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(loss_df["step"], loss_df["train_loss"], marker="o", markersize=3, linewidth=2, label="Train LogLoss")
    ax.plot(loss_df["step"], loss_df["test_loss"], marker="o", markersize=3, linewidth=2, label="Test LogLoss")
    ax.set_title(title)
    ax.set_xlabel("Iteration / Trees")
    ax.set_ylabel("LogLoss")
    ax.grid(alpha=0.25)
    ax.legend()
    plt.tight_layout()
    plt.show()

    display(loss_df.tail(10))


def extract_shap_values(shap_output):
    if isinstance(shap_output, list):
        return shap_output[1] if len(shap_output) > 1 else shap_output[0]

    values = shap_output.values if hasattr(shap_output, "values") else shap_output
    if isinstance(values, np.ndarray) and values.ndim == 3:
        return values[:, :, 1] if values.shape[2] > 1 else values[:, :, 0]

    return values


def plot_shap_test(model, X_test, model_name, sample_size=2000, max_display=20, random_state=42):
    if hasattr(X_test, "sample"):
        X_sample = X_test.sample(min(sample_size, len(X_test)), random_state=random_state).copy()
    else:
        rng = np.random.default_rng(random_state)
        idx = np.sort(rng.choice(len(X_test), size=min(sample_size, len(X_test)), replace=False))
        X_sample = X_test[idx]

    try:
        explainer = shap.TreeExplainer(model)
        shap_output = explainer.shap_values(X_sample)
    except Exception:
        explainer = shap.Explainer(model, X_sample)
        shap_output = explainer(X_sample)

    shap_values = extract_shap_values(shap_output)
    shap.summary_plot(shap_values, X_sample, show=False, max_display=max_display)
    plt.title(f"SHAP Summary - {model_name} (Test Sample)")
    plt.tight_layout()
    plt.show()

    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    shap_df = pd.DataFrame({
        "feature": X_sample.columns if hasattr(X_sample, "columns") else feature_cols,
        "mean_abs_shap": mean_abs_shap,
    }).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)

    print(f"Top {min(max_display, len(shap_df))} features by mean(|SHAP|) - {model_name}:")
    display(shap_df.head(max_display))


## 2. Load Data

In [ ]:
import joblib
import os

def load_data():
    base = os.path.join(os.path.dirname(os.getcwd()), "..", "src", "data", "processed")
    base = os.path.abspath(base)
    X_train = joblib.load(os.path.join(base, "X_train.pkl"))
    y_train = joblib.load(os.path.join(base, "y_train.pkl"))
    X_val   = joblib.load(os.path.join(base, "X_val.pkl"))
    y_val   = joblib.load(os.path.join(base, "y_val.pkl"))
    X_test  = joblib.load(os.path.join(base, "X_test.pkl"))
    y_test  = joblib.load(os.path.join(base, "y_test.pkl"))
    X_train_res = joblib.load(os.path.join(base, "X_train_res.pkl"))
    y_train_res = joblib.load(os.path.join(base, "y_train_res.pkl"))
    feature_cols = joblib.load(os.path.join(base, "feature_cols.pkl"))
    return X_train, y_train, X_val, y_val, X_test, y_test, X_train_res, y_train_res, feature_cols

X_train, y_train, X_val, y_val, X_test, y_test, X_train_res, y_train_res, feature_cols = load_data()

print(f"X_train_res: {X_train_res.shape}, fraud rate: {y_train_res.mean():.4f}")
print(f"X_val:       {X_val.shape},  fraud rate: {y_val.mean():.4f}")
print(f"X_test:      {X_test.shape}, fraud rate: {y_test.mean():.4f}")

## 3. Baseline CatBoost

In [ ]:
print("--- Training CatBoost Baseline ---")
cat_baseline = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    random_seed=SEED,
    eval_metric="AUC",
    verbose=0
)
cat_baseline.fit(X_train_res, y_train_res)
print("Done.")

## 4. Baseline Evaluation

In [ ]:
def calculate_metrics(y_true, y_pred, y_prob, label=""):
    metrics = {
        "F1":        f1_score(y_true, y_pred),
        "Recall":    recall_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "ROC-AUC":   roc_auc_score(y_true, y_prob),
        "PR-AUC":    average_precision_score(y_true, y_prob),
    }
    if label:
        print(f"\n=== {label} ===")
        for k, v in metrics.items():
            print(f"  {k}: {v:.4f}")
    return metrics

# Val
cat_val_preds = cat_baseline.predict(X_val)
cat_val_probs = cat_baseline.predict_proba(X_val)[:, 1]
cat_baseline_val = calculate_metrics(y_val, cat_val_preds, cat_val_probs, "CatBoost Baseline – Val")

# Test
cat_test_preds = cat_baseline.predict(X_test)
cat_test_probs = cat_baseline.predict_proba(X_test)[:, 1]
cat_baseline_test = calculate_metrics(y_test, cat_test_preds, cat_test_probs, "CatBoost Baseline – Test")

baseline_df = pd.DataFrame(
    [cat_baseline_val, cat_baseline_test],
    index=["CatBoost Val", "CatBoost Test"]
)
print("\nBaseline:")
display(baseline_df.round(4))

## 5. Optuna – Tối ưu Hyperparameters (100 trials, metric: PR-AUC trên val set)

In [ ]:
def cat_objective(trial):
    params = {
        "iterations":        trial.suggest_int("iterations", 200, 2000),
        "depth":             trial.suggest_int("depth", 4, 10),
        "learning_rate":     trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        "l2_leaf_reg":       trial.suggest_float("l2_leaf_reg", 1.0, 10.0),
        "subsample":         trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.5, 1.0),
        "min_data_in_leaf":  trial.suggest_int("min_data_in_leaf", 1, 50),
        "random_seed": SEED,
        "verbose": 0,
        "eval_metric": "AUC",
    }
    model = CatBoostClassifier(**params)
    model.fit(X_train_res, y_train_res)
    probs = model.predict_proba(X_val)[:, 1]
    return average_precision_score(y_val, probs)

cat_study = optuna.create_study(direction="maximize", study_name="CatBoost-PRAUC")
cat_study.optimize(cat_objective, n_trials=40, show_progress_bar=True)

print(f"\nCatBoost best PR-AUC (val): {cat_study.best_value:.4f}")
print("Best params:", cat_study.best_params)

## 5b. Retrain with best params


In [ ]:
print("--- Retraining CatBoost with best Optuna params ---")
cat_best = CatBoostClassifier(
    **cat_study.best_params,
    random_seed=SEED,
    verbose=0,
    eval_metric="AUC"
)
cat_best.fit(X_train_res, y_train_res)
print("Done.")

## 5c. Đánh giá sau Optuna – Val Set & so sánh Baseline

In [ ]:
cat_opt_val_preds = cat_best.predict(X_val)
cat_opt_val_probs = cat_best.predict_proba(X_val)[:, 1]

cat_opt_val = calculate_metrics(y_val, cat_opt_val_preds, cat_opt_val_probs, "CatBoost Optuna – Val")

val_comparison = pd.DataFrame(
    [cat_baseline_val, cat_opt_val],
    index=["CatBoost - Baseline", "CatBoost - Optuna"]
)
print("\nBaseline vs Optuna (Val Set):")
display(val_comparison.round(4))

cat_gain = cat_opt_val["PR-AUC"] - cat_baseline_val["PR-AUC"]
print(f"\nCatBoost PR-AUC gain: {cat_gain:+.4f}")

## 6. Đánh giá cuối cùng trên Test Set (chạy 1 lần)

In [ ]:
cat_opt_test_preds = cat_best.predict(X_test)
cat_opt_test_probs = cat_best.predict_proba(X_test)[:, 1]

cat_opt_test = calculate_metrics(y_test, cat_opt_test_preds, cat_opt_test_probs, "CatBoost Optuna – Test")

print("\nClassification Report – CatBoost Optuna (Test):")
print(classification_report(y_test, cat_opt_test_preds, target_names=["Non-fraud", "Fraud"]))

## 7. Visualization – Confusion Matrix & Precision-Recall Curve (sau Optuna)

In [ ]:
cm = confusion_matrix(y_test, cat_opt_test_preds)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Pred: 0', 'Pred: 1'],
    yticklabels=['True: 0', 'True: 1'],
    ax=ax
)
ax.set_title('Confusion Matrix – CatBoost', fontsize=14, pad=12)
ax.set_xlabel('Predicted Label')
ax.set_ylabel('True Label')
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"True Negatives  (TN): {tn}")
print(f"False Positives (FP): {fp}")
print(f"False Negatives (FN): {fn}")
print(f"True Positives  (TP): {tp}")

In [ ]:
# Precision-Recall Curve – CatBoost Baseline vs Optuna
plt.figure(figsize=(8, 6))

prec_b, rec_b, _ = precision_recall_curve(y_test, cat_test_probs)
prec_o, rec_o, _ = precision_recall_curve(y_test, cat_opt_test_probs)
auc_b = average_precision_score(y_test, cat_test_probs)
auc_o = average_precision_score(y_test, cat_opt_test_probs)

plt.plot(rec_b, prec_b, linestyle="--", label=f"CatBoost Baseline (PR-AUC={auc_b:.4f})", alpha=0.7)
plt.plot(rec_o, prec_o, linestyle="-",  label=f"CatBoost Optuna  (PR-AUC={auc_o:.4f})")

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve – CatBoost Baseline vs Optuna (Test Set)")
plt.legend(fontsize=9)
plt.grid(True)
plt.tight_layout()
plt.show()

# Optuna optimization history
plt.figure(figsize=(8, 4))
values = [t.value for t in cat_study.trials if t.value is not None]
best_so_far = pd.Series(values).cummax()
plt.plot(values, alpha=0.4, label="Trial PR-AUC")
plt.plot(best_so_far.values, color="red", label="Best so far")
plt.title("CatBoost – Optuna Optimization History")
plt.xlabel("Trial")
plt.ylabel("PR-AUC (Val)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## 8. Loss Curve


In [ ]:
# Dùng sample để plot nhanh hơn, tránh notebook chạy quá nặng
X_train_loss, y_train_loss = sample_xy(X_train_res, y_train_res, max_samples=30000)
X_test_loss, y_test_loss = sample_xy(X_test, y_test, max_samples=20000)


def catboost_loss_curve(model, X_train_eval, y_train_eval, X_test_eval, y_test_eval, max_points=60):
    total_steps = int(model.tree_count_)
    checkpoints = set(build_checkpoints(total_steps, max_points=max_points))
    rows = []

    train_iter = model.staged_predict_proba(X_train_eval)
    test_iter = model.staged_predict_proba(X_test_eval)

    for step, (train_prob, test_prob) in enumerate(zip(train_iter, test_iter), start=1):
        if step not in checkpoints:
            continue

        train_prob = np.clip(train_prob[:, 1], 1e-6, 1 - 1e-6)
        test_prob = np.clip(test_prob[:, 1], 1e-6, 1 - 1e-6)
        rows.append({
            "step": step,
            "train_loss": log_loss(y_train_eval, train_prob),
            "test_loss": log_loss(y_test_eval, test_prob),
        })

    return pd.DataFrame(rows)


cat_baseline_loss = catboost_loss_curve(cat_baseline, X_train_loss, y_train_loss, X_test_loss, y_test_loss)
plot_loss_curve(cat_baseline_loss, "CatBoost Baseline - Train vs Test LogLoss")

cat_optuna_loss = catboost_loss_curve(cat_best, X_train_loss, y_train_loss, X_test_loss, y_test_loss)
plot_loss_curve(cat_optuna_loss, "CatBoost Optuna - Train vs Test LogLoss")


## 9. SHAP Test


In [ ]:
plot_shap_test(cat_best, X_test, "CatBoost Optuna")


## 8. Feature Importance

In [ ]:
importance_df = pd.DataFrame({
    'feature':   feature_cols,
    'importance': cat_best.feature_importances_
}).sort_values('importance', ascending=False).reset_index(drop=True)

top_n = 20
top_df = importance_df.head(top_n)

fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(
    data=top_df, x='importance', y='feature',
    hue='feature', palette='viridis', legend=False, ax=ax
)
ax.set_title(f'Top {top_n} Feature Importances – CatBoost', fontsize=14)
ax.set_xlabel('Importance (Mean Decrease Impurity)')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

print(f"\nTop {top_n} features quan trá»ng nháº¥t:")
print(top_df.to_string(index=False))

## 9. Tóm tắt kết quả

In [ ]:
summary_df = pd.DataFrame(
    [cat_baseline_val, cat_baseline_test, cat_opt_val, cat_opt_test],
    index=[
        "CatBoost Baseline – Val",
        "CatBoost Baseline – Test",
        "CatBoost Optuna – Val",
        "CatBoost Optuna – Test"
    ]
)
print("\nFinal Summary:")
display(summary_df.round(4))